# Colab model prototype

This notebook was imported from upstream commit `04364a27596d8c3060f4eaa9e325b92db13add40`.
It loads public pretrained UCF and AASIST3 checkpoints. It contains no fine-tuning run or exported custom weights.

SENTINEL retains its existing pinned detectors. The UCF release matches this notebook, and the `MTUCI/AASIST3` alias currently resolves to the exact AASIST3 checkpoint already installed. See [the comparison](detector/COLAB_COMPARISON.md) for the saved results and integration decision.

These cells preserve the original prototype and diagnostic sequence. They depend on manually supplied media, private Google Drive files, and execution order. The secondary-channel response is simulated. The OpenAI Agents example and minimal tunnel API are not the desktop app's agent or detector server.

Saved outputs and credential literals have been removed. Optional legacy demo cells read `OPENAI_API_KEY` and `NGROK_AUTHTOKEN` from Colab Secrets. Previously committed credentials must be rotated by their owners. Do not reuse the old tunnel URL.

The desktop app uses `detector/server.py`, which supports the full health, audio, saved-media, and additional-evidence contracts. Its model runtime can use CUDA through `SENTINEL_DEVICE=cuda`; no accuracy improvement is established by moving inference to Colab.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                "transformers==4.52.4", "huggingface-hub==0.32.4", "tokenizers==0.21.1"], check=True)
print("Now: Runtime > Restart session, then continue from Cell 2.")

In [ ]:
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
if not os.path.exists("/content/AASIST3"):
    subprocess.run(["git", "clone", "https://github.com/mtuciru/AASIST3.git", "/content/AASIST3"], check=True)
sys.path.insert(0, "/content/AASIST3")

import torch, torchaudio
from model import aasist3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = aasist3.from_pretrained("MTUCI/AASIST3").to(device).eval()
print("MODEL_LOADED on", device)

def score_audio_file(path):
    waveform, sr = torchaudio.load(path)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)
    target_len = 64600
    if waveform.shape[0] < target_len:
        waveform = torch.nn.functional.pad(waveform, (0, target_len - waveform.shape[0]))
    else:
        waveform = waveform[:target_len]
    with torch.inference_mode():
        logits = model(waveform.unsqueeze(0).to(device))
        raw_probs = torch.softmax(logits, dim=1)
    # Confirmed fix: raw index 0 = spoof, index 1 = bonafide (inverted from the model card)
    return float(raw_probs[0, 0])  # spoof/risk probability

In [ ]:
from google.colab import files
print("Upload your real voice memo (say the demo line):")
uploaded = files.upload()
real_path = "/content/" + list(uploaded.keys())[0]

import os
os.system('pip install -q gtts')
from gtts import gTTS
gTTS(text="Hey it's Sarah, can you send me the deploy code, I'm locked out of my laptop").save("/content/fake_voice.mp3")
fake_path = "/content/fake_voice.mp3"

In [ ]:
print("REAL:", score_audio_file(real_path))
print("FAKE:", score_audio_file(fake_path))

In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.system('pip install -q openai-agents')
from agents import Agent, Runner, function_tool

@function_tool
def check_voice_authenticity(audio_path: str) -> dict:
    """Check whether an audio clip is likely an AI-generated/cloned voice."""
    risk = score_audio_file(audio_path)
    tier = "High" if risk > 0.75 else "Medium" if risk > 0.30 else "Low"
    return {"risk_score": round(risk, 3), "tier": tier}

@function_tool
def assess_request_sensitivity(request_text: str) -> dict:
    """Flag sensitive requests like credentials, code access, or transfers."""
    words = ["password", "code", "credential", "wire", "transfer", "deploy", "secret", "api key", "access"]
    matched = [w for w in words if w in request_text.lower()]
    return {"is_sensitive": len(matched) > 0, "matched_keywords": matched}

@function_tool
def verify_via_secondary_channel(contact_name: str, request_summary: str) -> dict:
    """Independently verify via a separate channel. Call only when voice risk
    is Medium/High AND the request is sensitive."""
    print(f"[SIMULATED ONLY: {contact_name} via Slack DM]: 'Did you just ask for: {request_summary}?'")
    return {"simulated": True, "contact_confirmed": False, "contact_reply": "No — I didn't send that message."}

agent = Agent(
    name="Signal Check Security Agent",
    instructions="""Always call check_voice_authenticity on the attached audio first.
Always call assess_request_sensitivity on the request text.
If voice risk is Medium or High AND the request is sensitive, you MUST call
verify_via_secondary_channel before doing anything else.
If the contact does not confirm, refuse and explain why plainly.
If everything checks out, approve and briefly explain your reasoning.
Always state the voice risk score/tier in your final answer.""",
    tools=[check_voice_authenticity, assess_request_sensitivity, verify_via_secondary_channel],
)

print("=== REAL VOICE ===")
result_real = await Runner.run(agent, f"Voice message from Sarah: 'Hey it's Sarah, can you send me the deploy code, I'm locked out of my laptop.' Audio file: {real_path}")
print(result_real.final_output)

print("=== SYNTHETIC VOICE ===")
result_fake = await Runner.run(agent, f"Voice message from Sarah: 'Hey it's Sarah, can you send me the deploy code, I'm locked out of my laptop.' Audio file: {fake_path}")
print(result_fake.final_output)

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

RECORD_JS = """
const sleep = t => new Promise(r => setTimeout(r, t));
var recorder, chunks = [];
async function record(sec) {
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(sec * 1000);
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  const blob = new Blob(chunks);
  const reader = new FileReader();
  const b64 = await new Promise(r => { reader.onloadend = () => r(reader.result); reader.readAsDataURL(blob); });
  return b64.split(',')[1];
}
"""

def record_audio(filename, seconds=8):
    display(Javascript(RECORD_JS))
    print(f"Recording for {seconds} seconds — speak your line now...")
    b64_data = eval_js(f'record({seconds})')
    with open(filename, 'wb') as f:
        f.write(b64decode(b64_data))
    size_kb = os.path.getsize(filename) / 1024
    print(f"Saved {filename} — {size_kb:.1f} KB")
    return filename

real_path = record_audio('/content/real_voice.webm', seconds=8)

In [ ]:
os.system('wget -q -O /content/fake_voice.wav "https://huggingface.co/amphion/vits_ljspeech/resolve/main/samples/Amphion_VITS_sample.wav"')
fake_path = "/content/fake_voice.wav"
print("Size:", os.path.getsize(fake_path) / 1024, "KB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import csv, tarfile, os
csv_path = '/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/flac_D_aa_labels.csv'
archive_path = '/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/flac_D_aa.tar'

if os.path.exists(csv_path):
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    bonafide_row = next(r for r in rows if r['label'] == 'bonafide')
    extract_dir = '/content/flac_D_aa'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(archive_path, 'r') as tf:
        for member in tf.getmembers():
            if member.name == bonafide_row['filename']:
                tf.extract(member, extract_dir, filter='data')
    real_path = f"{extract_dir}/{bonafide_row['filename']}"
    print("Using confirmed-real file:", real_path)
else:
    print("Drive path not found — this hackathon session may be under a different account. Use the LJSpeech fallback below instead.")

In [ ]:
print("REAL:", score_audio_file(real_path))
print("FAKE:", score_audio_file(fake_path))

In [ ]:
bonafide_rows = [r for r in rows if r['label'] == 'bonafide']
bonafide_row = bonafide_rows[5]  # any index other than 0 — different file, likely a normal, correctly-scored one

extract_dir = '/content/flac_D_aa'
with tarfile.open(archive_path, 'r') as tf:
    for member in tf.getmembers():
        if member.name == bonafide_row['filename']:
            tf.extract(member, extract_dir, filter='data')
real_path = f"{extract_dir}/{bonafide_row['filename']}"

print("REAL:", score_audio_file(real_path))
print("FAKE:", score_audio_file(fake_path))

In [ ]:
bonafide_row = bonafide_rows[15]  # try a different one
with tarfile.open(archive_path, 'r') as tf:
    for member in tf.getmembers():
        if member.name == bonafide_row['filename']:
            tf.extract(member, extract_dir, filter='data')
real_path = f"{extract_dir}/{bonafide_row['filename']}"
print("REAL:", score_audio_file(real_path))

In [ ]:
import os, subprocess

os.system('git clone https://github.com/SCLBD/DeepfakeBench.git /content/DeepfakeBench 2>/dev/null')
DFB = '/content/DeepfakeBench'

import numpy as np
np.sctypes = {
    'int': [np.int8, np.int16, np.int32, np.int64],
    'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
    'float': [np.float16, np.float32, np.float64, np.longdouble],
    'complex': [np.complex64, np.complex128, np.clongdouble],
    'others': [bool, object, bytes, str, np.void],
}

init_path = f'{DFB}/training/detectors/__init__.py'
lines_to_disable = [
    "from .utils import slowfast", "from .videomae_detector import VideoMAEDetector",
    "from .clip_detector import CLIPDetector", "from .timesformer_detector import TimeSformerDetector",
    "from .xclip_detector import XCLIPDetector", "from .ftcn_detector import FTCNDetector",
    "from .i3d_detector import I3DDetector", "from .altfreezing_detector import AltFreezingDetector",
    "from .stil_detector import STILDetector", "from .tall_detector import TALLDetector",
    "from .effort_detector import EffortDetector",
]
with open(init_path) as f: content = f.read()
for line in lines_to_disable:
    content = content.replace(line, f"# {line}")
with open(init_path, 'w') as f: f.write(content)

os.system("pip install -q fvcore iopath efficientnet_pytorch kornia imgaug lmdb dlib")

os.makedirs(f'{DFB}/preprocessing/dlib_tools', exist_ok=True)
os.system(f"wget -q -O {DFB}/preprocessing/dlib_tools/shape_predictor_81_face_landmarks.dat https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.0/shape_predictor_81_face_landmarks.dat")

os.makedirs(f'{DFB}/training/pretrained', exist_ok=True)
os.system(f"wget -q -O /tmp/pretrained.zip https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.0/pretrained.zip")
os.system(f"unzip -o -q /tmp/pretrained.zip -d {DFB}/training/pretrained")
import shutil
nested = f'{DFB}/training/pretrained/pretrained'
if os.path.exists(nested):
    for fn in os.listdir(nested):
        shutil.move(os.path.join(nested, fn), os.path.join(f'{DFB}/training/pretrained', fn))
    os.rmdir(nested)

os.makedirs(f'{DFB}/training/weights', exist_ok=True)
os.system(f"wget -q -O {DFB}/training/weights/ucf_best.pth https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.1/ucf_best.pth")

for name, path in {
    'dlib landmarks': f'{DFB}/preprocessing/dlib_tools/shape_predictor_81_face_landmarks.dat',
    'xception backbone': f'{DFB}/training/pretrained/xception-b5690688.pth',
    'ucf checkpoint': f'{DFB}/training/weights/ucf_best.pth',
}.items():
    ok = os.path.exists(path)
    print(f"{'OK' if ok else 'MISSING'}: {name}")

In [ ]:
import sys, os
sys.path.insert(0, f'{DFB}/training')
os.chdir(DFB)
from detectors import DETECTOR
import yaml, torch, torch.nn.functional as F
import dlib, cv2, numpy as np
from skimage import transform as trans

with open('training/config/detector/ucf.yaml') as f:
    config = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)
ckpt = torch.load('training/weights/ucf_best.pth', map_location=device)
model.load_state_dict(ckpt, strict=True)
model.eval()
print("UCF_LOADED on", device)

face_detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(f'{DFB}/preprocessing/dlib_tools/shape_predictor_81_face_landmarks.dat')

def crop_face(image_bgr, min_face_size=80):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces = face_detector(rgb, 1)
    if len(faces) == 0:
        return None
    face = max(faces, key=lambda r: r.width() * r.height())
    if face.width() < min_face_size or face.height() < min_face_size:
        return None
    shape = predictor(rgb, face)
    pts = np.array([[shape.part(i).x, shape.part(i).y] for i in [37, 44, 30, 49, 55]], dtype=np.float32)
    dst = np.array([[30.2946,51.6963],[65.5318,51.5014],[48.0252,71.7366],[33.5493,92.3655],[62.7299,92.2041]], dtype=np.float32)
    dst[:,0]+=8.0; dst*= (256/112.0)
    margin=256*0.15
    dst=(dst+margin)*256/(256+2*margin)
    tform = trans.SimilarityTransform(); tform.estimate(pts, dst)
    return cv2.warpAffine(rgb, tform.params[0:2,:], (256,256))

def score_image(image_bgr):
    cropped = crop_face(image_bgr)
    if cropped is None:
        return None
    arr = (cropped.astype(np.float32)/255.0 - 0.5)/0.5
    tensor = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(device)
    dummy = torch.zeros(1, dtype=torch.long).to(device)
    with torch.no_grad():
        pred = model({'image': tensor, 'label': dummy, 'mask': None, 'landmark': None}, inference=True)
    return float(F.softmax(pred['cls'], dim=1)[0,1])

# Reuse Celeb-DF-v2 test images already on this Drive account
import zipfile
zip_path = "/content/drive/MyDrive/Celeb-DF-v2.zip"
extract_dir = "/content/celebdf_sample"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    real_img = [n for n in names if n.startswith("Celeb-real/frames/id0_0000/")][0]
    fake_img = [n for n in names if n.startswith("Celeb-synthesis/frames/id0_id16_0000/")][0]
    zf.extract(real_img, extract_dir)
    zf.extract(fake_img, extract_dir)

print("REAL_VIDEO:", score_image(cv2.imread(f"{extract_dir}/{real_img}")))
print("FAKE_VIDEO:", score_image(cv2.imread(f"{extract_dir}/{fake_img}")))

In [ ]:
@function_tool
def check_video_authenticity(image_path: str) -> dict:
    """Check whether a video frame/image shows a real or AI-generated/manipulated face."""
    img = cv2.imread(image_path)
    risk = score_image(img)
    if risk is None:
        return {"error": "No face detected in this image"}
    tier = "High" if risk > 0.75 else "Medium" if risk > 0.30 else "Low"
    return {"risk_score": round(risk, 3), "tier": tier}

In [ ]:
agent = Agent(
    name="Signal Check Security Agent",
    instructions="""You handle incoming requests that may include a voice message
and/or a video frame from a call.
If audio is attached, call check_voice_authenticity on it.
If a video frame is attached, call check_video_authenticity on it.
Always call assess_request_sensitivity on the request text.
If either voice risk or video risk is Medium or High AND the request is
sensitive, you MUST call verify_via_secondary_channel before doing anything else.
If the contact does not confirm, refuse and explain why plainly.
If everything checks out, approve and briefly explain your reasoning.
State every risk score/tier you checked in your final answer.""",
    tools=[check_voice_authenticity, check_video_authenticity, assess_request_sensitivity, verify_via_secondary_channel],
)

print("=== REAL VOICE + REAL VIDEO ===")
print((await Runner.run(agent, f"Video call message from Sarah: 'Hey it's Sarah, can you send me the deploy code, I'm locked out of my laptop.' Audio file: {real_path}. Video frame: {extract_dir}/{real_img}")).final_output)

print("=== FAKE VOICE + FAKE VIDEO ===")
print((await Runner.run(agent, f"Video call message from Sarah: 'Hey it's Sarah, can you send me the deploy code, I'm locked out of my laptop.' Audio file: {fake_path}. Video frame: {extract_dir}/{fake_img}")).final_output)

In [ ]:
video_model = model  # currently holds UCF, correctly captured before it gets overwritten
model = aasist3.from_pretrained("MTUCI/AASIST3").to(device).eval()  # restore AASIST3 as 'model'
print("Audio model restored, video model kept separately as 'video_model'")

def score_image(image_bgr):
    cropped = crop_face(image_bgr)
    if cropped is None:
        return None
    arr = (cropped.astype(np.float32)/255.0 - 0.5)/0.5
    tensor = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(device)
    dummy = torch.zeros(1, dtype=torch.long).to(device)
    with torch.no_grad():
        pred = video_model({'image': tensor, 'label': dummy, 'mask': None, 'landmark': None}, inference=True)
    return float(F.softmax(pred['cls'], dim=1)[0,1])

# Confirm both are independently correct now
print("AUDIO_CHECK:", score_audio_file(real_path))
print("VIDEO_CHECK:", score_image(cv2.imread(f"{extract_dir}/{real_img}")))

In [ ]:
import transformers, huggingface_hub, tokenizers
print(transformers.__version__, huggingface_hub.__version__, tokenizers.__version__)

In [ ]:
os.system('pip install -q fastapi uvicorn nest-asyncio pyngrok')
import nest_asyncio; nest_asyncio.apply()
import threading, uvicorn, base64, cv2, numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

class AnalyzeRequest(BaseModel):
    jpegBase64: str
    capturedAt: int

class AnalyzeResponse(BaseModel):
    deepfakeProbability: float
    faceDetected: bool
    confidence: Optional[float] = None
    model: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/analyze")
def analyze(request: AnalyzeRequest) -> AnalyzeResponse:
    jpeg_bytes = base64.b64decode(request.jpegBase64)
    arr = np.frombuffer(jpeg_bytes, dtype=np.uint8)
    image_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    cropped = crop_face(image_bgr)  # your existing function from earlier
    if cropped is None:
        return AnalyzeResponse(deepfakeProbability=0.0, faceDetected=False, confidence=None, model="ucf-xception")
    prob = score_image(image_bgr)  # your existing function from earlier
    return AnalyzeResponse(deepfakeProbability=float(prob), faceDetected=True, confidence=None, model="ucf-xception")

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
print("Local server up on :8000")

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
public_url = ngrok.connect(8000, bind_tls=True)
print("DETECTOR_URL =", f"{public_url}/analyze")

In [ ]:
# Waveform sanity - rule out "both files are somehow decoding to the same broken thing"
wf_real, sr_real = torchaudio.load(real_path)
wf_fake, sr_fake = torchaudio.load(fake_path)
print("REAL waveform: shape=", wf_real.shape, "sr=", sr_real, "mean=", wf_real.mean().item(), "std=", wf_real.std().item())
print("FAKE waveform: shape=", wf_fake.shape, "sr=", sr_fake, "mean=", wf_fake.mean().item(), "std=", wf_fake.std().item())

# Raw logits - the actual diagnostic signal
def get_raw_logits(path):
    waveform, sr = torchaudio.load(path)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)
    target_len = 64600
    if waveform.shape[0] < target_len:
        waveform = torch.nn.functional.pad(waveform, (0, target_len - waveform.shape[0]))
    else:
        waveform = waveform[:target_len]
    with torch.inference_mode():
        return model(waveform.unsqueeze(0).to(device))

print("REAL raw logits:", get_raw_logits(real_path))
print("FAKE raw logits:", get_raw_logits(fake_path))

# Check every submodule actually is in eval mode, not just the top-level object
still_training = [name for name, m in model.named_modules() if m.training]
print("Submodules still in train() mode (should be empty):", still_training)

In [ ]:
for name, param in model.named_parameters():
    if 'feature_extractor' in name or 'wav2vec' in name.lower():
        print(name, "mean=", param.data.mean().item(), "std=", param.data.std().item())
        break

In [ ]:
print("Model loaded:", 'model' in globals())
print("crop_face defined:", 'crop_face' in globals())

In [ ]:
import requests
print(requests.get(f"{public_url}/health", timeout=10).json())
